In [2]:
import sys
from pathlib import Path
project_root = "/home/velocitatem/Documents/Projects/PHANTOM/experiments"
if str(Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()) not in sys.path:
  sys.path.insert(0, str(project_root))

In [3]:
import pandas as pd

In [5]:
from procesing.steps import (
    FetchInteractionsStep,
    FetchPriceLogsStep,
    FetchExperimentsStep,
    JoinExperimentsStep,
    CreatePriceBucketsStep,
    AugmentEventNamesStep,
    ChunkByTimeWindowStep,
    ComputeDemandForChunksStep,
    AggregatePriceLogsStep,
    ComputeElasticityStep,
    FitPricingFunctionStep,
    PredictPricesStep,
)
from procesing.context import PipelineContext
from procesing.providers import SupabaseProvider, BackendAPIProvider

In [6]:
class Provider(SupabaseProvider, BackendAPIProvider):
    def __init__(self, backend_url: str):
        SupabaseProvider.__init__(self)
        BackendAPIProvider.__init__(self, backend_url=backend_url)
# example run
context = PipelineContext(
    provider=Provider(backend_url="http://localhost:5000"),
    store_mode='hotel',
    window_size='15min',

)


In [7]:
df=FetchInteractionsStep(context).transform(None)
df.head()

,sessionId,experimentId,eventName,page,productId,storeMode,userAgent,ts,metadata_referrer,metadata_elementText,metadata_dateIndex,metadata_dwellTime,metadata_type,metadata_roomType,metadata_price,metadata_nights,metadata_total,metadata_itemCount,dateIndex
0,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,page_view,/,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-11-25T20:20:13.061Z,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
1,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,hover_over_title,/hotel/products,d018efc1-25e9-4284-b276-80386e048b25,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-11-25T20:21:17.425Z,NaN,Junior Suite,1.0,1200.0,NaN,NaN,NaN,NaN,NaN,NaN,1
2,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,hover_over_paragraph,/hotel/products,d018efc1-25e9-4284-b276-80386e048b25,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-11-25T20:21:19.496Z,NaN,price,1.0,1202.0,NaN,NaN,NaN,NaN,NaN,NaN,1
3,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,page_view,/hotel/products/d018efc1-25e9-4284-b276-80386e...,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-11-25T20:21:21.922Z,http://localhost:3000/hotel/products?dateIndex...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
4,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,learn_more_about_item,/hotel/products/d018efc1-25e9-4284-b276-80386e...,d018efc1-25e9-4284-b276-80386e048b25,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-11-25T20:21:22.674Z,NaN,NaN,1.0,NaN,hotel,Junior Suite,NaN,NaN,NaN,NaN,1


In [8]:
df = CreatePriceBucketsStep(context).transform(df)
df.head()

,sessionId,experimentId,eventName,page,productId,storeMode,userAgent,ts,metadata_referrer,metadata_elementText,metadata_dateIndex,metadata_dwellTime,metadata_type,metadata_roomType,metadata_price,metadata_nights,metadata_total,metadata_itemCount,dateIndex,price_bucket
0,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,page_view,/,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-11-25T20:20:13.061Z,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,
1,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,hover_over_title,/hotel/products,d018efc1-25e9-4284-b276-80386e048b25,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-11-25T20:21:17.425Z,NaN,Junior Suite,1.0,1200.0,NaN,NaN,NaN,NaN,NaN,NaN,1,
2,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,hover_over_paragraph,/hotel/products,d018efc1-25e9-4284-b276-80386e048b25,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-11-25T20:21:19.496Z,NaN,price,1.0,1202.0,NaN,NaN,NaN,NaN,NaN,NaN,1,
3,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,page_view,/hotel/products/d018efc1-25e9-4284-b276-80386e...,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-11-25T20:21:21.922Z,http://localhost:3000/hotel/products?dateIndex...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,
4,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,learn_more_about_item,/hotel/products/d018efc1-25e9-4284-b276-80386e...,d018efc1-25e9-4284-b276-80386e048b25,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-11-25T20:21:22.674Z,NaN,NaN,1.0,NaN,hotel,Junior Suite,NaN,NaN,NaN,NaN,1,


In [9]:
df = AugmentEventNamesStep(context).transform(df)
df.tail()

,sessionId,experimentId,eventName,page,productId,storeMode,userAgent,ts,metadata_referrer,metadata_elementText,...,metadata_dwellTime,metadata_type,metadata_roomType,metadata_price,metadata_nights,metadata_total,metadata_itemCount,dateIndex,price_bucket,metadata_schema
78,c404dbe5-116f-42c0-b199-503516dbbe91,fd01774c-f629-4bcb-88b8-c818856af72a,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...,2025-11-29T17:32:45.064Z,,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,,
79,c404dbe5-116f-42c0-b199-503516dbbe91,fd01774c-f629-4bcb-88b8-c818856af72a,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...,2025-11-29T18:13:53.858Z,,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,,
80,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,53aefd07-f66a-4d7f-ba8b-7ea1fc562d35,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-12-04T11:13:15.884Z,,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,,
81,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,53aefd07-f66a-4d7f-ba8b-7ea1fc562d35,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-12-04T11:18:53.473Z,,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,,
82,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,53aefd07-f66a-4d7f-ba8b-7ea1fc562d35,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-12-04T11:19:05.094Z,,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,,


In [10]:
price_df=FetchPriceLogsStep(context).fit_transform(None)
price_df.tail()

,productId,price,sessionId,experimentId,storeMode,ts
213,2cd7f756-fc65-4ba0-ab01-74521c1fff43,100.0,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,hotel,2025-12-04T11:18:56.320Z
214,2ddabbfc-4127-48fc-86dc-ebc4c677efa2,100.0,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,hotel,2025-12-04T11:19:05.434Z
215,2cd7f756-fc65-4ba0-ab01-74521c1fff43,100.0,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,hotel,2025-12-04T11:19:05.338Z
216,2cd7f756-fc65-4ba0-ab01-74521c1fff43,100.0,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,hotel,2025-12-04T11:19:05.597Z
217,2ddabbfc-4127-48fc-86dc-ebc4c677efa2,100.0,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,None,hotel,2025-12-04T11:19:05.594Z


In [11]:
df_chunks = ChunkByTimeWindowStep(context).transform(df)

In [12]:
len(df_chunks)

11

In [13]:
df_chunks[-1]['window_start']

Timestamp('2025-12-04 11:15:00+0000', tz='UTC')

In [14]:
df_chunks[-1]['window_end']

Timestamp('2025-12-04 11:30:00+0000', tz='UTC')

In [15]:
df_chunks[-1]['data'].head()

,sessionId,experimentId,eventName,page,productId,storeMode,userAgent,ts,metadata_referrer,metadata_elementText,...,metadata_dwellTime,metadata_type,metadata_roomType,metadata_price,metadata_nights,metadata_total,metadata_itemCount,dateIndex,price_bucket,metadata_schema
81,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,53aefd07-f66a-4d7f-ba8b-7ea1fc562d35,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-12-04 11:18:53.473000+00:00,,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,,
82,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,53aefd07-f66a-4d7f-ba8b-7ea1fc562d35,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-12-04 11:19:05.094000+00:00,,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,,


In [16]:
demand = ComputeDemandForChunksStep(context).transform(df_chunks)
len(demand)

11

In [17]:
demand[-1]['demand_vector']

,productId,demand_score
0,bec37f41-7756-47ae-9219-f5854290f4e7,0
1,5e666c06-023a-415b-9976-be0956bbc405,0
2,d018efc1-25e9-4284-b276-80386e048b25,0
3,2cd7f756-fc65-4ba0-ab01-74521c1fff43,0
4,51266ddb-5b07-47b7-89ee-5b5cae94bb11,0
...,...,...
79,0d1c9a3a-bc37-4417-a59f-de4b994944cb,0
80,fc64bd74-4dfa-4f78-802a-39d6aa4c39fe,0
81,d85d4c52-baa0-435f-81ac-b0c27a5251b3,0
82,93bc00e5-8cfe-42af-8322-49bc27407688,0


In [18]:
price_df_agg = AggregatePriceLogsStep(context).transform(price_df)

In [19]:
price_df_agg[-1]['price_vector']

,productId,price
0,2cd7f756-fc65-4ba0-ab01-74521c1fff43,100.00
1,2ddabbfc-4127-48fc-86dc-ebc4c677efa2,100.00
2,51266ddb-5b07-47b7-89ee-5b5cae94bb11,100.00
3,d018efc1-25e9-4284-b276-80386e048b25,100.00
4,aaae8177-0803-4421-8702-f3ffeeeadcd9,389.04
5,7f71fbe2-343c-4a46-94ea-07cbd903a86c,327.94
6,d6affcb8-6616-47f8-af14-2ec8583f0781,391.43
7,0fbcf915-ecf1-4ec3-9b00-8bbc314e2a81,900.97
8,eceedfb3-ec52-4453-9aab-88dd9a6b6ca3,640.54


In [20]:
elasticity = ComputeElasticityStep(context).transform((demand, price_df_agg))
elasticity

,productId,elasticity,std_error,n_obs
0,d018efc1-25e9-4284-b276-80386e048b25,-0.222054,0.447102,11
1,2cd7f756-fc65-4ba0-ab01-74521c1fff43,-0.072857,0.510130,11
2,51266ddb-5b07-47b7-89ee-5b5cae94bb11,-0.291083,0.346879,11
3,2ddabbfc-4127-48fc-86dc-ebc4c677efa2,-10.223968,0.000000,11
4,7f71fbe2-343c-4a46-94ea-07cbd903a86c,0.000000,0.000000,9
...,...,...,...,...
79,0d1c9a3a-bc37-4417-a59f-de4b994944cb,0.000000,0.000000,0
80,fc64bd74-4dfa-4f78-802a-39d6aa4c39fe,0.000000,0.000000,0
81,d85d4c52-baa0-435f-81ac-b0c27a5251b3,0.000000,0.000000,0
82,93bc00e5-8cfe-42af-8322-49bc27407688,0.000000,0.000000,0


In [21]:
elasticity.set_index('productId')

,elasticity,std_error,n_obs
productId,,,
d018efc1-25e9-4284-b276-80386e048b25,-0.222054,0.447102,11
2cd7f756-fc65-4ba0-ab01-74521c1fff43,-0.072857,0.510130,11
51266ddb-5b07-47b7-89ee-5b5cae94bb11,-0.291083,0.346879,11
2ddabbfc-4127-48fc-86dc-ebc4c677efa2,-10.223968,0.000000,11
7f71fbe2-343c-4a46-94ea-07cbd903a86c,0.000000,0.000000,9
...,...,...,...
0d1c9a3a-bc37-4417-a59f-de4b994944cb,0.000000,0.000000,0
fc64bd74-4dfa-4f78-802a-39d6aa4c39fe,0.000000,0.000000,0
d85d4c52-baa0-435f-81ac-b0c27a5251b3,0.000000,0.000000,0


In [22]:
elasticity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   productId   84 non-null     object 
 1   elasticity  84 non-null     float64
 2   std_error   84 non-null     float64
 3   n_obs       84 non-null     int64  
dtypes: float64(2), int64(1), object(1)
memory usage: 2.8+ KB


In [23]:
df['productId'] = df['productId'].astype(str)
elasticity['productId'] = elasticity['productId'].astype(str)
dff=df.join(elasticity.set_index('productId'), how="left", on="productId")

In [24]:
dff.tail()

,sessionId,experimentId,eventName,page,productId,storeMode,userAgent,ts,metadata_referrer,metadata_elementText,...,metadata_price,metadata_nights,metadata_total,metadata_itemCount,dateIndex,price_bucket,metadata_schema,elasticity,std_error,n_obs
78,c404dbe5-116f-42c0-b199-503516dbbe91,fd01774c-f629-4bcb-88b8-c818856af72a,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...,2025-11-29T17:32:45.064Z,,NaN,...,NaN,NaN,NaN,NaN,<NA>,,,NaN,NaN,NaN
79,c404dbe5-116f-42c0-b199-503516dbbe91,fd01774c-f629-4bcb-88b8-c818856af72a,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...,2025-11-29T18:13:53.858Z,,NaN,...,NaN,NaN,NaN,NaN,<NA>,,,NaN,NaN,NaN
80,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,53aefd07-f66a-4d7f-ba8b-7ea1fc562d35,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-12-04T11:13:15.884Z,,NaN,...,NaN,NaN,NaN,NaN,<NA>,,,NaN,NaN,NaN
81,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,53aefd07-f66a-4d7f-ba8b-7ea1fc562d35,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-12-04T11:18:53.473Z,,NaN,...,NaN,NaN,NaN,NaN,<NA>,,,NaN,NaN,NaN
82,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,53aefd07-f66a-4d7f-ba8b-7ea1fc562d35,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-12-04T11:19:05.094Z,,NaN,...,NaN,NaN,NaN,NaN,<NA>,,,NaN,NaN,NaN


In [25]:
experiments = FetchExperimentsStep(context).transform(dff)
experiments.tail()

,id,subject_name,xp_human_only,xp_market_mode,xp_task_id,task
0,53aefd07-f66a-4d7f-ba8b-7ea1fc562d35,Daniel,False,hotel,517b8078-cf4c-4a1f-b943-75281c69a5b3,"{'task_name': 'Cheapest Room', 'task_def_of_do..."
1,d10f5ab3-a7b7-4e97-8d94-ab06f1537c0a,Full Agent,False,hotel,517b8078-cf4c-4a1f-b943-75281c69a5b3,"{'task_name': 'Cheapest Room', 'task_def_of_do..."
2,fd01774c-f629-4bcb-88b8-c818856af72a,Daniel 1,True,hotel,920c3deb-18c6-4586-bbc4-4ce4d1ae6f2d,"{'task_name': 'Cheapest Room w/ View', 'task_d..."


In [26]:
dff_exp = JoinExperimentsStep(context).transform((dff,experiments))

In [27]:
dff_exp.tail()

,sessionId,experimentId,eventName,page,productId,storeMode,userAgent,ts,metadata_referrer,metadata_elementText,...,elasticity,std_error,n_obs,exp_subject,exp_human_only,exp_market_mode,exp_task_id,task_name,task_def_of_done,task_description
76,c404dbe5-116f-42c0-b199-503516dbbe91,fd01774c-f629-4bcb-88b8-c818856af72a,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...,2025-11-29T17:32:45.064Z,,NaN,...,NaN,NaN,NaN,Daniel 1,True,hotel,920c3deb-18c6-4586-bbc4-4ce4d1ae6f2d,Cheapest Room w/ View,User added to cart a the cheapest room of all ...,Find the cheapest room with a nice view in the...
77,c404dbe5-116f-42c0-b199-503516dbbe91,fd01774c-f629-4bcb-88b8-c818856af72a,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...,2025-11-29T18:13:53.858Z,,NaN,...,NaN,NaN,NaN,Daniel 1,True,hotel,920c3deb-18c6-4586-bbc4-4ce4d1ae6f2d,Cheapest Room w/ View,User added to cart a the cheapest room of all ...,Find the cheapest room with a nice view in the...
78,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,53aefd07-f66a-4d7f-ba8b-7ea1fc562d35,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-12-04T11:13:15.884Z,,NaN,...,NaN,NaN,NaN,Daniel,False,hotel,517b8078-cf4c-4a1f-b943-75281c69a5b3,Cheapest Room,A room was added and purchased.,Find the cheapest hotel room in multiple steps...
79,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,53aefd07-f66a-4d7f-ba8b-7ea1fc562d35,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-12-04T11:18:53.473Z,,NaN,...,NaN,NaN,NaN,Daniel,False,hotel,517b8078-cf4c-4a1f-b943-75281c69a5b3,Cheapest Room,A room was added and purchased.,Find the cheapest hotel room in multiple steps...
80,d423ce8a-77aa-4c9a-94d4-d1adddcc3472,53aefd07-f66a-4d7f-ba8b-7ea1fc562d35,page_view,/hotel/products,None,hotel,Mozilla/5.0 (X11; Linux x86_64; rv:145.0) Geck...,2025-12-04T11:19:05.094Z,,NaN,...,NaN,NaN,NaN,Daniel,False,hotel,517b8078-cf4c-4a1f-b943-75281c69a5b3,Cheapest Room,A room was added and purchased.,Find the cheapest hotel room in multiple steps...
